# PolyUQ_Sensi: sensitivity post-processing for the guyed-mast beam model

This notebook post-processes pre-computed HPC propagation results for the
`UQ_Modal_FEM` beam model. It cannot run end-to-end without the published
refodat dataset (DOI 10.71758/refodat.46). Set the environment variable
`POLYUQ_BEAM_DATA_DIR` to the local path of the unpacked dataset before
running.


In [ ]:
%matplotlib widget
%load_ext autoreload
%autoreload 2
%load_ext snakeviz

import sys
import os
import logging
import matplotlib
# a bug in jupyter / ipympl / matplotlib needs this here when using %maptlotlib widget
# somehow rc_context is broken in that case
matplotlib.rc('text.latex', preamble=r"\usepackage{siunitx}\usepackage{xfrac}\usepackage{amssymb, amsfonts}")
# to adjust figure uncomment here and comment out %matplotlib widget, restart kernel and 
# make sure jupyterlab is running in a x-forwarded and connected ssh session and the right display is set
# os.environ['DISPLAY'] = 'localhost:17.0'
# matplotlib.use("qtagg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from polyuq import *
from helpers import get_pcd

%aimport -sys -logging -matplotlib -matplotlib.pyplot -numpy -pandas - scipy.stats -scipy.stats.qmc -ray

In [ ]:
from examples.UQ_Modal_FEM import mapping_function, vars_definition, est_imp, opt_inc

In [ ]:
vars_ale, vars_epi, arg_vars = vars_definition()
dim_ex = 'cartesian'

# %%snakeviz
N_mcs_ale = 13717 # N_mcs = 1e6 = N_mcs_ale * N_mcs_epi
N_mcs_epi = 729 # = 3^6 = 2.56^n_imp ~ 3^n_imp -> cover every corner and midpoints in a full-factorial design (but distributed)
use_dm = True
result_dir = os.environ.get('POLYUQ_BEAM_DATA_DIR', '.')


In [ ]:
ret_name = ['damp_freqs','zetas','frf'][0]
if ret_name == 'frf':
    ret_ind = {'frequencies':105, 'space':2}
else:
    ret_ind = {'modes':7}
ret_dir = f'{ret_name}-{".".join(str(e) for e in ret_ind.values())}'
samp_path = os.path.join(result_dir,'polyuq_samp.npz')
prop_path = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_prop.npz')

poly_uq = PolyUQ(vars_ale, vars_epi, dim_ex=dim_ex)
poly_uq.load_state(samp_path, differential='samp')
poly_uq.load_state(prop_path, differential='prop')

### Considerations

we have a matrix of input samples N_mcs_ale x n_vars_ale, N_mcs_epi x n_vars_epi
we have a matrix of output samples N_mcs_ale x N_mcs_epi x 1

#### choices:

1) compute sensitivities for vars_ale and vars_epi separately for selected samples of the other kind, respectively -> allows second-order effects?
2) compute sensitivities from the diagonal of the output matrix -> following regular qMCS sampling
3) flatten outputs and compute sensitivities -> would problems appear due to repetition of values?
    - might even be good for bootstrapping

#### first-order sensitivity:
compute variance of output

- for each variable:
    - argsort input samples
    - subdivide output and input samples into s subdivisions
    - for each subdivision:
        - compute mean
    - compute variance of mean -> first-order sensitivity 
    - error/confidence measure? -> use the bootstrap method
    

#### total-order sensitivity
compute mean over all but one input

- fix all but one input 
    - grouping into n-1 dimensional hypercubes, that capture the variance of input j but other inputs remain relatively constant (not possible from existing samples)
    - compute variance

Develop it on a Sobol's g-function and verify it on the Ishigami function, then transfer it to polyuq

- define the functions as python functions
- generate and propagate qMC samples and regular MC samples
- write a function for variance based sensi
- verify and validate
- (implement density based sensi)
- generate sample lattices as in polyuq and test different options for choosing samples
- transfer to polyuq
- implement confidence bounds with the bootstrap method

### Implementaiton

In [ ]:
import scipy.stats.qmc
import scipy.stats
import numpy as np
import SALib
from SALib.analyze import delta as delta

In [ ]:
def sobol_g(X, a):
    num_samples, num_params = X.shape
    assert len(a) == num_params
    Y = np.ones((num_samples,))
    for k in range(num_params):
        Y *= (np.abs(4 * X[:,k] - 2) + a[k])/(1 + a[k])
    return Y

def sobol_g_S_an(a):
    num_params = a.shape[0]
    denom = 1
    for k in range(num_params):
        beta = 1 / 3 * (1 + a[k])**(-2)
        denom *= (1 + beta)
    denom -= 1
    S_an = np.empty((num_params,))
    for k in range(num_params):
        beta = 1 / 3 * (1 + a[k])**(-2)
        S_an[k] = beta / denom
    return S_an

def ishigami(X, a):
    return np.sin(X[:,0]) + a[0] * np.sin(X[:,1])**2 + a[1] * X[:,2]**4 * np.sin(X[:,0])

def ishigami_S_an(a):
    V = np.full((3,), 1 / (a[0]**2 / 8 + a[1] * np.pi**4 / 5 + a[1]**2 * np.pi**8 / 18 + 1 / 2))
    V[0] *= a[1] * np.pi**4 / 5 + a[1]**2 * np.pi**8 / 50 + 1 / 2
    V[1] *= a[0]**2 / 8
    V[2]  = 0
    return V
    

In [ ]:
def estimate_first_order_sens(x, y, num_subdivisions=None):
    num_samples, num_params = x.shape
    if num_subdivisions is None:
        # taken from SALib.analyze.delta
        exp = 2.0 / (7.0 + np.tanh((1500.0 - num_samples) / 500.0))
        num_subdivisions = int(np.round(min(int(np.ceil(num_samples**exp)), 48)))
        # print(f'Using {num_subdivisions} subdivisions')
    S = np.full((num_params,), 1/np.var(y))
    n_per_s = num_samples // num_subdivisions
    # print(f'Remaining {num_samples % num_subdivisions} samples will be discarded.')
    for k in range(num_params):
        sort_ind = np.argsort(x[:,k])
        # xk_sort = x[sort_ind,k]
        y_sort = y[sort_ind]
        means = np.empty((num_subdivisions,))
        for s in range(num_subdivisions):
            means[s] = np.mean(y_sort[s * n_per_s:(s + 1) * n_per_s])
        S[k] *= np.var(means)
    return S

In [ ]:
if True:
    num_params = 10
    a = np.random.random((num_params,))
    # a = [0,0.9,99.9]
    obj_fun = sobol_g
    an_fun = sobol_g_S_an
else:
    num_params = 3
    a = np.random.random((num_params,))
    obj_fun = ishigami
    an_fun = ishigami_S_an
problem = {'num_vars': num_params, 'names': [f'x{k}' for k in range(num_params)]}
print(an_fun(a))

In [ ]:
N_mcs = 732
engine = scipy.stats.qmc.Halton(num_params)
X = engine.random(N_mcs)
if obj_fun == ishigami:
    X -= 0.5
    X *= 2*np.pi

Y = obj_fun(X,a)

In [ ]:
# assume 2 ale params, remaining epi
N_mcs_epi = 729
N_mcs_ale = 13717
engine = scipy.stats.qmc.Halton(num_params)
X_gen = engine.random(max(N_mcs_epi, N_mcs_ale))
if obj_fun == ishigami:
    X_gen -= 0.5
    X_gen *= 2*np.pi

X_lat = np.empty((N_mcs_ale, N_mcs_epi, num_params))

for var_ale in range(2):
    X_lat[:,:,var_ale] = np.repeat(X_gen[:N_mcs_ale,var_ale:var_ale + 1],N_mcs_epi,1)
for var_epi in range(2, num_params):
    X_lat[:,:,var_epi] = np.repeat(X_gen[:N_mcs_epi,var_epi:var_epi + 1].T,N_mcs_ale,0)
# display(X_lat[0,0,:])
# display(X_lat[0,1,:])
# display(X_lat[0,2,:])
X_lat_flat = X_lat.reshape((N_mcs_ale*N_mcs_epi, num_params)) # appends rows to each other
# display(X_lat[-1,-3,:])
# display(X_lat[-1,-2,:])
# display(X_lat[-1,-1,:])
Y_lat_flat = obj_fun(X_lat_flat, a)

In [ ]:
if False: #diagonal selection
    ind = np.eye(N_mcs_ale, N_mcs_epi, dtype=bool).reshape((N_mcs_ale*N_mcs_epi,))
    X = X_lat_flat[ind,:]
    Y = Y_lat_flat[ind]
elif False: # select all, flat; only variance-based is fast enough for this
    X = X_lat_flat
    Y = Y_lat_flat
elif True: # select a random subset (with replacement)
    N_mcs_min = min(N_mcs_ale, N_mcs_epi)
    ind = np.random.choice(np.arange(Y_lat_flat.size),N_mcs_min,)
    X = X_lat_flat[ind,:]
    Y = Y_lat_flat[ind]

In [ ]:
S = estimate_first_order_sens(X,Y)
S_an = an_fun(a)

In [ ]:
SA_S = delta.analyze(problem, X, Y)
SA_S

In [ ]:
plt.figure()
plt.plot(S, ls='none',marker='x')
plt.plot(S_an, ls='none',marker='+')
plt.plot(SA_S['S1'], ls='none',marker='o',fillstyle='none')
plt.ylim((0,1))

### Convergence study

- define the MAE for both measures (sum of absolute difference)
- Generate the convergence for increasing N_mcs (in both dimensions with variable ale-to-epi-ratio) N_mcs = 2**n (n=8...15)
- Consider cases: different test functions/num_params, different re-sampling strategies,  

In [ ]:
def MAE(choice,meth,):
    problem = {'num_vars': num_params, 'names': [f'x{k}' for k in range(num_params)]}
    all_n = np.arange(8,15)
    # MAE_V = np.empty((all_n.size,))
    # MAE_D = np.empty((all_n.size,))
    MAE = np.empty((all_n.size,))
    n_ind = np.arange(all_n.size)
    np.random.shuffle(n_ind)
    for i in n_ind:
        n = all_n[i]             
        N_mcs = 2**n
        
        #assuming some random ratio of epistemic to aleatory samples
        ea_rat = np.random.random(1) + 1
        if choice==2:
            N_mcs_epi = int(np.sqrt(N_mcs / ea_rat))
            N_mcs_ale = int(np.sqrt(ea_rat * N_mcs))
        else:
            N_mcs_epi = N_mcs
            N_mcs_ale = int(ea_rat * N_mcs)
        
        engine = scipy.stats.qmc.Halton(num_params)
        X_gen = engine.random(max(N_mcs_epi, N_mcs_ale))
        if obj_fun == ishigami:
            X_gen -= 0.5
            X_gen *= 2 * np.pi
        
        X_lat = np.empty((N_mcs_ale, N_mcs_epi, num_params), dtype=np.float32)

        # assume 2 ale params, remaining epi
        for var_ale in range(2):
            X_lat[:,:,var_ale] = np.repeat(X_gen[:N_mcs_ale,var_ale:var_ale + 1],N_mcs_epi,1)
        for var_epi in range(2, num_params):
            X_lat[:,:,var_epi] = np.repeat(X_gen[:N_mcs_epi,var_epi:var_epi + 1].T,N_mcs_ale,0)
        X_lat_flat = X_lat.reshape((N_mcs_ale*N_mcs_epi, num_params)) # appends rows to each other
        # Y_lat_flat = obj_fun(X_lat_flat, a)
        if choice==1: #diagonal selection
            ind = np.eye(N_mcs_ale, N_mcs_epi, dtype=bool).reshape((N_mcs_ale*N_mcs_epi,))
            X = X_lat_flat[ind,:]
            # Y = Y_lat_flat[ind]
        elif choice==2: # select all, flat; only variance-based is fast enough for this
            X = X_lat_flat
            # Y = Y_lat_flat
        elif choice==3: # select a random subset (with replacement)
            N_mcs_min = min(N_mcs_ale, N_mcs_epi)
            ind = np.random.choice(np.arange(X_lat_flat.shape[0]),N_mcs_min,)
            X = X_lat_flat[ind,:]
            # Y = Y_lat_flat[ind]
        Y = obj_fun(X, a)
        S_an = an_fun(a)
        
        if meth=='var':
            S = estimate_first_order_sens(X,Y)
        elif meth=='dens':
            S = delta.analyze(problem, X, Y)['S1']
        
        MAE[i] = np.mean(np.abs(S - S_an))
    return choice, meth, MAE

In [ ]:
import ray

In [ ]:
ray.shutdown()
ray.init(address='auto')
futures1 = []
remote_MAE = ray.remote(num_cpus=4, )(MAE)

In [ ]:
for choice in [1,2,3]:
    for meth in ['var','dens']:
        for i in range(30):
            futures1.append(remote_MAE.remote(choice,meth))        

In [ ]:
MAE_dens_1 = []
MAE_dens_2 = []
MAE_dens_3 = []
MAE_var_1 = []
MAE_var_2 = []
MAE_var_3 = []

In [ ]:
### run that for sobol-g 10, sobol-g 3, ishigami

futures = set(futures1)
while True:
    ready, wait = ray.wait(
        list(futures), num_returns=min(len(futures), 10), timeout=60)
    finished = []
    failed = []
    for obj_ref in ready:
        try:
            choice, meth, thisMAE = ray.get(obj_ref)
            if meth=='dens':
                all_MAE = [None,MAE_dens_1,MAE_dens_2,MAE_dens_3][choice]
            elif meth=='var':
                all_MAE = [None,MAE_var_1,MAE_var_2,MAE_var_3][choice]
            else:
                raise RuntimeErrror
            all_MAE.append(thisMAE)
            finished.append(obj_ref)
        except ray.exceptions.RayTaskError as e:
            print(e)
            failed.append(obj_ref)
    
    size_before = len(futures)
    futures.difference_update(finished)
    futures.difference_update(failed)
    print(f"Finished {len(finished)}, failed {len(failed)} samples. Remaining {len(futures)} samples. (before {size_before})")
        
    if len(futures) == 0:
        break

In [ ]:
mean_MAE_dens_1 = np.mean(np.array(MAE_dens_1), axis=0)
mean_MAE_dens_2 = np.mean(np.array(MAE_dens_2), axis=0)
mean_MAE_dens_3 = np.mean(np.array(MAE_dens_3), axis=0)
mean_MAE_var_1 = np.mean(np.array(MAE_var_1), axis=0)
mean_MAE_var_2 = np.mean(np.array(MAE_var_2), axis=0)
mean_MAE_var_3 = np.mean(np.array(MAE_var_3), axis=0)

In [ ]:
plt.figure()
x = 2**np.arange(8,15)
plt.plot(x,mean_MAE_dens_1, label='dens, diag')
plt.plot(x,mean_MAE_dens_2, label='dens, all')
plt.plot(x,mean_MAE_dens_2, label='dens, choice')
plt.plot(x,mean_MAE_var_1, label='var, diag')
plt.plot(x,mean_MAE_var_2, label='var, all')
plt.plot(x,mean_MAE_var_2, label='var, choice')
plt.legend()

In [ ]:
mean_MAE_dens_3, mean_MAE_var_3, mean_MAE_dens_2, mean_MAE_var_2

### Confidence

In [ ]:
Nr = 4096

def bootstr_wrap(ind):
    ind = np.random.randint(0, Y_lat_flat.shape[0], Nr)
    X = X_lat_flat[ind,:]
    Y = Y_lat_flat[ind]
    return estimate_first_order_sens(X,Y)

In [ ]:
res = scipy.stats.bootstrap(np.arange(Nr)[np.newaxis,:], bootstr_wrap, vectorized=False)
ind = np.random.randint(0, Y_lat_flat.shape[0], Nr)
S1_ = estimate_first_order_sens(X_lat_flat[ind,:],Y_lat_flat[ind])

In [ ]:
# SA_S = delta.analyze(problem, X_lat_flat, Y_lat_flat, y_resamples=100, method='all')
np.savetxt('polyuq_results/X.txt',X_lat_flat)
np.savetxt('polyuq_results/Y.txt',Y_lat_flat)
# SA_S

In [ ]:
# S1_ = S1
with matplotlib.rc_context(get_pcd('print')):
    plt.figure()
    handles = []
    S1 =(res.confidence_interval.high+res.confidence_interval.low)/2
    # plt.plot(np.arange(num_params)-0.15, S1_,marker='x', ls='none', color='red'
    #         )
    eb_cont = plt.errorbar(np.arange(num_params), S1, [S1-res.confidence_interval.low,res.confidence_interval.high-S1], 
                           marker='+', ls='none', capsize=3, label='$S_{1,i}$ (variance-based)', color='dimgrey')
    eb_cont = plt.errorbar(np.arange(num_params),SA_S['delta'], SA_S['delta_conf'], 
                           marker='+', ls='none', capsize=3, label='$\delta_i$ (density-based)', color='lightgrey')
    # eb_cont = plt.errorbar(np.arange(num_params)+0.15,SA_S['S1'], SA_S['S1_conf'], 
    #                        marker='x', ls='none', capsize=3, label='$\S_{1,i}$ (density-based)', color='lightgrey')
    lines = plt.plot(np.arange(num_params),an_fun(a), ls='none', 
                     marker='x', fillstyle='none', label='$S_{1,i}$ (Analytical)', color='black')
    # get handles
    handles, labels = plt.gca().get_legend_handles_labels()
    # remove the errorbars
    handles = [h[0] if isinstance(h, tuple) else h for h in handles]
    # use them in the legend
    plt.gca().legend(handles, labels)#, loc='upper left',numpoints=1)
    plt.gca().set_xticks(np.arange(num_params), [f'$p_{{{i+1}}}$' for i in range(num_params)])
    plt.ylim(ymin=0)
    plt.xlabel('Parameters')
    plt.ylabel('Sensitivity')
    plt.subplots_adjust(top=0.97,bottom=0.11, left=0.1, right=0.97)
    # plt.savefig(f'figures/ex_sensi_conf_k10.png')
    # plt.savefig(f'figures/ex_sensi_conf_k10.pdf')

How to proceed from here? What do we need for the thesis?
- Convergence Study is in the papers and the choice of a sample selection method is deemed a minor detail, which must not be proven by a convergence study
- An example with sobol-g 10 parameters, showing confidence intervals with variance-based, from density based and analytical results

- port the methods to PolyUQ
- run Sensi for uq_modal_beam and include confidence intervals as a table in the journalpaper (not another graph)

In [ ]:
ind = np.random.randint(0, Y_lat_flat.shape[0], Nr)
SA_S = delta.analyze(problem, X_lat_flat[ind,:], Y_lat_flat[ind], y_resamples=Nr, method='all')
SA_S.plot()

In [ ]:
ind = np.random.randint(0, Y_lat_flat.shape[0], 4*Nr)
SA_S = delta.analyze(problem, X_lat_flat[ind,:], Y_lat_flat[ind], y_resamples=Nr, method='all')
SA_S.plot()

In [ ]:
ind = np.random.randint(0, Y_lat_flat.shape[0], 4*Nr)
SA_S = delta.analyze(problem, X_lat_flat[ind,:], Y_lat_flat[ind], y_resamples=Nr, method='all')
SA_S.plot()

In [ ]:
SA_S = delta.analyze(problem, X_lat_flat, Y_lat_flat, y_resamples=100)
SA

### PolyUQ Integration

In [ ]:
logger= logging.getLogger('polyuq.polymorphic_uncertainty')
logger.setLevel(level=logging.DEBUG)
res = poly_uq.estimate_sensi('var', y_resamples=10000)
display(res)

In [ ]:
# poly_uq.save_state(os.path.join(result_dir,'polyuq_sens_var.npz'), differential='sensi')
poly_uq.load_state(os.path.join(result_dir,'estimations', f'{ret_dir}/polyuq_sens_var.npz'), differential='sensi')
print(poly_uq.S_point, poly_uq.S_conf, poly_uq.S_meth)
display(res)

In [ ]:
plt.figure()
S1 = res[0]
plt.errorbar(np.arange(len(res[2])), S1, [S1 - res[1][0], res[1][1] - S1], 
                           marker='+', ls='none', capsize=3, label='$S_{1,i}$ (variance-based)', color='dimgrey')
plt.gca().set_xticks(np.arange(len(res[2])), res[2])

In [ ]:
res = poly_uq.estimate_sensi('dens', y_resamples=10000)

In [ ]:
plt.figure()
S1 = res[0]
plt.errorbar(np.arange(len(res[2])), S1, [S1-res[1][0],res[1][1]-S1], 
                           marker='+', ls='none', capsize=3, label='$S_{1,i}$ (variance-based)', color='dimgrey')
plt.gca().set_xticks(np.arange(len(res[2])), res[2])
None

Ok that seems to be working and we get plausible results. Now whats next:
- look at the other quantities
    - side by side or stacked bar plot /w all 14  sensititivies (per output quantity)
    - heatmap out quant vs. parameters (Si could be color, confidence could be alpha encoded)
- for the paper, 
    - we don't want another graph, or maybe we should have one?
    - prefer tabular or aggregated data
    - ...?
- for the thesis
    - write up more details about SA?!
    - use the top graphs in the uq_modal_beam example



In [ ]:
def sensi(result_dir, ret_name, ret_ind, meth, Nr):
    ret_dir = f'{ret_name}-{".".join(str(e) for e in ret_ind.values())}'
    samp_path = os.path.join(result_dir,'polyuq_samp.npz')
    prop_path = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_prop.npz')

    poly_uq = PolyUQ(vars_ale, vars_epi, dim_ex=dim_ex)
    poly_uq.load_state(samp_path, differential='samp')
    poly_uq.load_state(prop_path, differential='prop')

    poly_uq.estimate_sensi(meth, y_resamples=Nr)
    poly_uq.save_state(os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_sens_{meth}.npz'), differential='sens')

In [ ]:
import ray

In [ ]:
ray.shutdown()

In [ ]:
ray.init(address='auto')
futures = []
remote_sensit = ray.remote(num_cpus=1)(sensi)

In [ ]:
ret_name = ['damp_freqs','zetas','frf'][1]

y_resamples=10000
if ret_name == 'frf':
    inds = range(10*3)
else:
    inds = range(14)
    
for meth in ['var', 'dens']:
    for ind in inds:
        if ret_name == 'frf':
            ret_ind = {'frequencies':ind//3, 'space':ind%3}
            if ind%3==0:
                continue
        else:
            ret_ind = {'modes':ind}
        robj = remote_sensit.remote(result_dir, ret_name, ret_ind, meth, y_resamples)

In [ ]:
ret_name = ['damp_freqs','zetas','frf'][0:2]
meth='var'
if ret_name == 'frf':
    inds = range(10*3)
elif isinstance(ret_name, str):
    inds = range(14)
else:
    inds = range(10)
# plt.figure()

if isinstance(ret_name, str):
    ret_names = [ret_name]
else:
    ret_names = ret_name
    
S_point = np.empty((len(inds)*len(ret_names),9))
S_conf_intv = np.empty((len(inds)*len(ret_names),9))
names = []
for i,ret_name in enumerate(ret_names):
    for ind in inds:
        if ret_name == 'frf':
            ret_ind = {'frequencies':ind//3, 'space':ind%3}
            if ind%3==0:
                continue
        else:
            ret_ind = {'modes':ind}
        ret_dir = f'{ret_name}-{".".join(str(e) for e in ret_ind.values())}'
        samp_path = os.path.join(result_dir,'polyuq_samp.npz')
        prop_path = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_prop.npz')
        sens_path = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_sens_{meth}.npz')

        poly_uq = PolyUQ(vars_ale, vars_epi, dim_ex=dim_ex)
        poly_uq.load_state(samp_path, differential='samp')
        poly_uq.load_state(prop_path, differential='prop')
        poly_uq.load_state(sens_path, differential='sens')

        S_point[i*len(inds)+ind,:] = poly_uq.S_point
        S_conf_intv[i*len(inds)+ind,:] = np.diff(poly_uq.S_conf, axis=0)
    names = list(poly_uq.inp_samp_prim.columns)
    
#     plt.errorbar(np.arange(len(names))+ind/15, S_point, [S_point-S_conf[0],S_conf[1]-S_point], 
#                            marker='+',markersize=1, ls='none', capsize=1, label=ret_dir)
# plt.gca().set_xticks(np.arange(len(names)), names)   
# remove Ice occurence
ind_I_occ = np.array(names)!='ice_occ'
S_point = S_point[:,ind_I_occ]
S_conf_intv = S_conf_intv[:,ind_I_occ]
names = np.array(names)[ind_I_occ]

In [ ]:
names = list(poly_uq.inp_samp_prim.columns)
    
#     plt.errorbar(np.arange(len(names))+ind/15, S_point, [S_point-S_conf[0],S_conf[1]-S_point], 
#                            marker='+',markersize=1, ls='none', capsize=1, label=ret_dir)
# plt.gca().set_xticks(np.arange(len(names)), names)   
# remove Ice occurence
ind_I_occ = np.array(names)!='ice_occ'
S_point = S_point[:,ind_I_occ]
S_conf_intv = S_conf_intv[:,ind_I_occ]
names = np.array(names)[ind_I_occ]

In [ ]:
im_ratio = S_point.shape[1]/S_point.shape[0]
pcd = get_pcd('print')
pcd['figure.figsize'] = (pcd['figure.figsize'][0],pcd['figure.figsize'][0]*im_ratio)
with matplotlib.rc_context(pcd):
    plt.figure()
    S_conf_alpha = np.copy(S_conf_intv.T)/S_point.T
    S_conf_alpha /= S_conf_alpha.max()
    S_conf_alpha *= -1 
    S_conf_alpha += 1
    plt.imshow(S_point.T, 
               #alpha=S_conf_alpha, 
               vmin=0, vmax=1,
              cmap='Greys')
    plt.colorbar(fraction=0.0465*im_ratio, pad=0.04, label='Sensitivity $S_1$')
    if len(ret_names)>1:
        plt.gca().set_xticks(np.arange(2*len(inds)), [f'$f_{{{i+1}}}$' for i in range(len(inds))]+[f'$\zeta_{{{i+1}}}$' for i in range(len(inds))])
    elif ret_name == 'zetas':        
        plt.gca().set_xticks(np.arange(len(inds)), [f'$\zeta_{{{i+1}}}$' for i in range(len(inds))])
    elif ret_name == 'damp_freqs':
        plt.gca().set_xticks(np.arange(len(inds)), [f'$f_{{{i+1}}}$' for i in range(len(inds))])
    names_dict = {'N_wire':'$N_\mathrm{cbl}$',
                  'dD':'$d_\mathrm{D}$',
                  'ice_occ':'$\mathfrak{I}_\mathrm{occ}$',
                  'b':'$b$',
                  't':'$t$',
                  'add_mass':'$m_\mathrm{add}$',
                  'A_wire':'$A_\mathrm{cbl}$',
                  'zeta':'$\zeta_\mathrm{glob}$',
                  'ice_mass':'$\mathfrak{I}_\mathrm{m}$'}
    plt.gca().set_yticks(np.arange(len(names)), [names_dict[name] for name in names])   
    plt.xlabel('Output parameter')
    plt.ylabel('Input parameter')
    # plt.clabel('Sensivity')
    if len(ret_names)>1:
        plt.subplots_adjust(top=0.97, bottom=0.14, left=0.1, right=0.92)
        plt.axvline(len(inds)-0.5, c='k',lw=1)
        plt.savefig(f'figures/sensi_matrix_all_{meth}.png')
        plt.savefig(f'figures/sensi_matrix_all_{meth}.pdf')
    else:
        plt.subplots_adjust(top=0.97, bottom=0.09, left=0.1, right=0.92)
        plt.savefig(f'figures/sensi_matrix_{ret_name}_{meth}.png')
        plt.savefig(f'figures/sensi_matrix_{ret_name}_{meth}.pdf')

In [ ]:
plt.subplots_adjust(top=0.97, bottom=0.14, left=0.1, right=0.92)

In [ ]:
plt.figure()
plt.hist((np.copy(S_conf_intv.T)).flatten(), bins=500)

In [ ]:
(np.copy(S_conf_intv.T)).flatten().mean()

In [ ]:
plt.close('all')
ret_name = 'zetas'
meth='var'
# plt.figure()
fig, axes = plt.subplots(14,9, sharex='col',sharey='row')
for mode in range(14):
    
    axes[mode,0].set_ylabel(mode)
    ret_ind = {'modes':mode}
    ret_dir = f'{ret_name}-{".".join(str(e) for e in ret_ind.values())}'
    samp_path = os.path.join(result_dir,'polyuq_samp.npz')
    prop_path = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_prop.npz')
    sens_path = os.path.join(result_dir, 'estimations', f'{ret_dir}/polyuq_sens_{meth}.npz')

    poly_uq = PolyUQ(vars_ale, vars_epi, dim_ex=dim_ex)
    poly_uq.load_state(samp_path, differential='samp')
    poly_uq.load_state(prop_path, differential='prop')
    poly_uq.load_state(sens_path, differential='sens')

    print(poly_uq.S_point, poly_uq.inp_samp_prim.columns)

    self = poly_uq

    vars_epi = self.vars_epi
    vars_ale = self.vars_ale

    N_mcs_ale = self.N_mcs_ale
    N_mcs_epi = self.N_mcs_epi

    inp_samp = self.inp_samp_prim
    out_samp = self.out_samp

    inds_ale, inds_epi = np.mgrid[0:N_mcs_ale, 0:N_mcs_epi]
    inds_ale, inds_epi = inds_ale.ravel(), inds_epi.ravel()

    names_ale = [var.name for var in vars_ale if var.primary]
    names_epi = [var.name for var in vars_epi if var.primary]

    arrays_grid  = [inp_samp[name].iloc[inds_ale] for name in names_ale]
    arrays_grid += [inp_samp[name].iloc[inds_epi] for name in names_epi]

    X = np.array(arrays_grid).T
    Y = out_samp[inds_ale, inds_epi]
    names = names_ale + names_epi

    ind_occ = names.index('ice_occ')
    ind_mass = names.index('ice_mass')
    X[:,ind_mass] *= X[:,ind_occ] 

    ind = np.random.randint(Y.shape[0], size=1000)

    for i in range(len(names)):
        axes[mode,i].plot(X[ind,i], Y[ind], marker=',', ls='none')
for i in range(len(names)):
    axes[0,i].set_title(names[i])